# Run one experiment (colab)

1. Run the first two cells (they prepare code, data and pretrained weights; safe to re-run).
2. Set `RUN_ID` and `SEED` below (the list shows every ID and whether it is done), then run the rest.

- Code: upload the whole `polypssl` folder to `My Drive/polypssl/code/polypssl/` (replace the old one).
- Results: `My Drive/polypssl/outputs/<ID>__<name>/seed<k>/` (the notebook links `/content/outputs` to it), so
  a disconnect loses at most 500 iterations: just run all cells again with the same `RUN_ID` and `SEED`.

Each run folder ends with `model_final.pth` (kept), `log.csv`, `config.yaml`, `per_image.csv`, `summary.csv`;
`outputs/ALL_RESULTS.csv` collects every finished run. Finished runs are never retrained.

In [ ]:
import os
import subprocess
import shutil
import sys
import zipfile
from pathlib import Path, PurePosixPath

from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

PLATFORM = "colab"
WORK = Path("/content")
CODE_DIR = WORK / "polypssl"
CODE_SOURCE = Path("/content/drive/MyDrive/polypssl/code/polypssl")
CODE_ZIP = Path("/content/drive/MyDrive/polypssl/code/polypssl.zip")

def extract_code_archive(archive, destination):
    with zipfile.ZipFile(archive) as zf:
        names = [PurePosixPath(info.filename.replace("\\", "/")) for info in zf.infolist() if not info.is_dir()]
        if any(path.is_absolute() or ".." in path.parts or
               (path.parts and path.parts[0].endswith(":")) for path in names):
            raise ValueError(f"Unsafe path found in {archive}")
        prefix = PurePosixPath("polypssl")
        if not any(path == prefix / "train.py" for path in names):
            prefix = PurePosixPath(".")
        if not any(path == prefix / "train.py" for path in names):
            raise FileNotFoundError("The code ZIP must contain train.py and the polypssl package")
        for info in zf.infolist():
            path = PurePosixPath(info.filename.replace("\\", "/"))
            if info.is_dir() or "__MACOSX" in path.parts:
                continue
            try:
                relative = path.relative_to(prefix) if str(prefix) != "." else path
            except ValueError:
                continue
            if not relative.parts:
                continue
            target = destination.joinpath(*relative.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as source, target.open("wb") as output:
                shutil.copyfileobj(source, output)

if (CODE_SOURCE / "train.py").is_file() and (CODE_SOURCE / "polypssl").is_dir():
    shutil.copytree(CODE_SOURCE, CODE_DIR, dirs_exist_ok=True)
    print(f"Copied project folder from Drive: {CODE_SOURCE}")
elif CODE_ZIP.is_file():
    extract_code_archive(CODE_ZIP, CODE_DIR)
    print(f"Extracted project ZIP from Drive: {CODE_ZIP}")
else:
    raise FileNotFoundError(
        "Project not found. Upload the polypssl folder to My Drive/polypssl/code/polypssl "
        "or place polypssl.zip in My Drive/polypssl/code.")
os.environ["POLYPSSL_ENV"] = PLATFORM
os.environ["POLYPSSL_DATA"] = "/content/data"
OUTPUT_DIR = Path("/content/outputs")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/polypssl/outputs")
if not OUTPUT_DIR.exists() and not OUTPUT_DIR.is_symlink():
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.symlink_to(DRIVE_OUTPUT_DIR, target_is_directory=True)
elif OUTPUT_DIR.is_symlink() and OUTPUT_DIR.resolve() != DRIVE_OUTPUT_DIR.resolve():
    raise RuntimeError(f"Refusing unexpected outputs symlink: {OUTPUT_DIR} -> {OUTPUT_DIR.resolve()}")
elif not OUTPUT_DIR.is_symlink():
    raise RuntimeError(f"{OUTPUT_DIR} is a normal folder, so results would NOT be saved to Drive. "
                       "Use Runtime > Disconnect and delete runtime, then run this notebook again.")
os.environ["POLYPSSL_OUT"] = str(OUTPUT_DIR)

if not (CODE_DIR / "train.py").is_file():
    raise FileNotFoundError(f"train.py was not prepared in {CODE_DIR}")
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
print(f"[{PLATFORM}] code: {CODE_DIR}")
print(f"[{PLATFORM}] data: {os.environ['POLYPSSL_DATA']}")
print(f"[{PLATFORM}] outputs: {os.environ['POLYPSSL_OUT']}")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python setup_env.py

In [ ]:
!python run.py --list --seed 0

## Choose the run

In [ ]:
RUN_ID = "C1"     # e.g. C1, B2, D1, E1  (see the list above and experiments.yaml)
SEED = 0

In [ ]:
!python run.py --run {RUN_ID} --seed {SEED}

## Results

In [ ]:
from IPython.display import Image, display
from polypssl.paths import get_locations

folder = next(get_locations().out.glob(f"{RUN_ID}__*/seed{SEED}"), None)
print("run folder:", folder)
if folder is not None and (folder / "report.txt").exists():
    print((folder / "report.txt").read_text())
    for png in sorted(folder.glob("*.png")):
        display(Image(filename=str(png)))

In [ ]:
!python run.py --results

## Only after a new test set (e.g. PolypGen) is added

Evaluates every finished model again; nothing is retrained (a few minutes per model). Remove the `#` to run it.

In [ ]:
# !python run.py --run all --reevaluate --seed {SEED}